## CREATING CO + SLAB


In [4]:
from ase.build import fcc100
from ase.constraints import FixAtoms
from ase import Atoms
from ase.io import write
import numpy as np

# ── 1. Cu(100) slab 4x4x4 ───────────────────────────────────
slab = fcc100("Cu", size=(4, 4, 4), vacuum=15.0)
slab.center(vacuum=15.0, axis=2)

fixed_ids = [a.index for a in slab if a.position[2] < slab.cell[2][2] / 2 - 5]
slab.set_constraint(FixAtoms(indices=fixed_ids))

a_Cu  = 3.634
z_top = slab.positions[:, 2].max()

# ── 2. Pick one top-site anchor───

top_atoms  = [a for a in slab if abs(a.position[2] - z_top) < 0.1]
top_sorted = sorted(top_atoms, key=lambda a: (a.position[0], a.position[1]))
anchor1    = top_sorted[5]          # central-ish atom
x1, y1    = anchor1.position[:2]




# ── 3. Two CO molecules, upright on top sites ────────────────
CO_bond  = 1.15   # Å — gas-phase CO (C down, O up)
C_z_above = 1.30  # Å — C height above top Cu layer

CO1 = Atoms("CO", positions=[
    [x1, y1, z_top + C_z_above],
    [x1, y1, z_top + C_z_above + CO_bond],
])


slab_co = slab + CO1 

# ── 4. Sanity checks ─────────────────────────────────────────
n  = len(slab)
p  = slab_co.positions
C1, O1 = p[n],   p[n+1]



co1_dist = np.linalg.norm(O1 - C1)

CO1.set_cell(slab.get_cell())

print("=" * 52)
print("  *CO on Cu(100) — geometry summary")
print("=" * 52)
print(f"  CO1 bond length          : {co1_dist:.3f} Å  (target 1.15 Å)")
print(f"  Total atoms              : {len(slab_co)}")
print("=" * 52)


# ── 5. Write outputs ─────────────────────────────────────────
write("slab_CO.xyz", slab_co)
write("slab_CO.cif", slab_co)
write("slab.xyz", slab)
write("slab.cif", slab)
write("CO.xyz", CO1)
write("CO.cif", CO1)

print("\n  Written: slab_CO.xyz")
print("           slab_CO.cif")

print("\n  Written: slab.xyz")
print("           slab.cif")
print("\n  Written: CO.xyz")
print("           CO.cif")



  *CO on Cu(100) — geometry summary
  CO1 bond length          : 1.150 Å  (target 1.15 Å)
  Total atoms              : 66

  Written: slab_CO.xyz
           slab_CO.cif

  Written: slab.xyz
           slab.cif

  Written: CO.xyz
           CO.cif


## CO dimer on top

In [20]:


# ============================================================
#  Build *C2O2^- dimer on Cu(100) for DFT geometry optimisation
#  Product of: *CO + CO(g) + e^- --> *C2O2^-  (rate-det. step)
#
#  Geometry from Fig. 3b of Calle-Vallejo & Koper, ACIE 2013:
#  ASYMMETRIC top-bridge configuration:
#    C1 (lower, 2.03 e-): top site, bonded to surface Cu
#    O1 (bridge, 7.48 e-): bends toward adjacent Cu (bridge contact)
#    C2 (upper, 3.52 e-): tilted upward, no direct surface contact
#    O2 (free,  7.91 e-): points toward vacuum
#
#  The lower CO bends its O toward the surface (bridge contact),
#  the upper CO tilts away — NOT a symmetric bridge-bridge geometry.
# ============================================================
 
from ase.build import fcc100
from ase.constraints import FixAtoms
from ase import Atoms
from ase.io import write
from ase.visualize import view
import numpy as np
 
# ── 1. Cu(100) slab ─────────────────────────────────────────
slab = fcc100("Cu", size=(4, 4, 4), vacuum=15.0)
slab.center(vacuum=15.0, axis=2)
 
fixed_ids = [a.index for a in slab if a.position[2] < slab.cell[2][2] / 2 - 5]
slab.set_constraint(FixAtoms(indices=fixed_ids))
 
a_Cu  = 3.634
z_top = slab.positions[:, 2].max()
 
# ── 2. Anchor Cu atom (top site for C1) ─────────────────────
top_atoms  = [a for a in slab if abs(a.position[2] - z_top) < 0.1]
top_sorted = sorted(top_atoms, key=lambda a: (a.position[0], a.position[1]))
anchor     = top_sorted[4]
x0, y0    = anchor.position[:2]
 
# ── 3. *C2O2^- asymmetric geometry (Fig. 3b) ────────────────
#
#  C1: top site, ~1.3 Å above anchor Cu
C1 = np.array([x0, y0, z_top + 1.30])
#
#  O1: bridge contact — C1-O1 tilted ~55° from vertical toward +x
#      → O1 ends up ~0.6 Å above surface (bridge Cu contact)
C1O1_tilt = np.radians(90)
O1 = C1 + 1.22 * np.array([np.sin(C1O1_tilt), 0.0, -np.cos(C1O1_tilt)])
#
#  C2: upper arm — C1-C2 axis tilted ~40° from vertical toward -x
CC_tilt = np.radians(40)
C2 = C1 + 1.46 * np.array([-np.sin(CC_tilt), 0.0, np.cos(CC_tilt)])
#
#  O2: terminal, continues the C1→C2 direction (points to vacuum)
C2O2_dir = (C2-C1)/np.linalg.norm(C2-C1)
O2 = C2 + 1.18 * C2O2_dir
 
# ── 4. Add dimer to slab ─────────────────────────────────────
dimer      = Atoms(symbols=["C", "C", "O", "O"],
                   positions=[C1, C2, O1, O2])

print ("pos ",dimer.positions)
dimer.translate([0, a_Cu/2, 0])
print ("pos ",dimer.positions)
slab_dimer = slab + dimer

dimer.set_cell(slab.get_cell())
dimer.center()
 
# ── 5. Sanity checks ─────────────────────────────────────────
n   = len(slab)
p   = slab_dimer.positions
C1p, C2p, O1p, O2p = p[n], p[n+1], p[n+2], p[n+3]
 
cc  = np.linalg.norm(C2p - C1p)
co1 = np.linalg.norm(O1p - C1p)
co2 = np.linalg.norm(O2p - C2p)
c1z = C1p[2] - z_top
o1z = O1p[2] - z_top
c2z = C2p[2] - z_top
o2z = O2p[2] - z_top
 
print("=" * 56)
print("  *C2O2^- dimer — geometry summary (pre-relaxation)")
print("=" * 56)
print(f"  C–C  bond length         : {cc:.3f} Å   (target 1.46 Å)")
print(f"  C1–O1 bond length        : {co1:.3f} Å   (target ~1.22 Å)")
print(f"  C2–O2 bond length        : {co2:.3f} Å   (target ~1.18 Å)")
print(f"  C1 height above surface  : {c1z:.3f} Å   (top site)")
print(f"  O1 height above surface  : {o1z:.3f} Å   (bridge contact, should be < 1 Å)")
print(f"  C2 height above surface  : {c2z:.3f} Å   (upper arm)")
print(f"  O2 height above surface  : {o2z:.3f} Å   (free, toward vacuum)")
print(f"  Total atoms              : {len(slab_dimer)}")
print(f"  Fixed atoms (DFT)        : {len(fixed_ids)}")
print("=" * 56)
print()
print("  Bader charge reference (paper Fig. 3b):")
print("    C1 (surface): 2.03 e-  |  O1 (bridge): 7.48 e-")
print("    C2 (upper)  : 3.52 e-  |  O2 (free)  : 7.91 e-")
print("    Sum ≈ 21 e-  →  dimer carries ~1 extra electron → C2O2^-")
print("=" * 56)
 
assert 1.30 < cc  < 1.65, f"C-C = {cc:.3f} Å out of range"
assert 1.10 < co1 < 1.40, f"C1-O1 = {co1:.3f} Å out of range"
assert 1.10 < co2 < 1.40, f"C2-O2 = {co2:.3f} Å out of range"
assert o1z < 1.4,          f"O1 too high ({o1z:.2f} Å) — bridge contact not formed"
assert c2z > c1z,          f"C2 should be higher than C1"
 
# ── 6. Write outputs ─────────────────────────────────────────
write("slab_C2O2_dimer_top.xyz", slab_dimer)
write("slab_C2O2_dimer_top.cif", slab_dimer)

write("C2O2_dimer.xyz",  dimer)   # for CP2K / visualisation
 
print("\n  Written: slab_C2O2_dimer.xyz  (CP2K / ASE input)")
print("\n  Written: C2O2_dimer.xyz  (CP2K / ASE input)")
print("           slab_C2O2_dimer.cif  (VESTA visualisation)")
 
# ── 7. Visualise ─────────────────────────────────────────────
# view(slab_dimer)
 
# ── 8. CP2K hints ────────────────────────────────────────────
print("""
  ── CP2K setup ──────────────────────────────────────────────
  &DFT
    CHARGE  -1          ! C2O2^- (one extra electron)
    MULTIPLICITY  2     ! doublet — check spin contamination
  &END DFT
 
  GEO_OPT:
    MAX_FORCE  0.05     ! eV/Å — same threshold as *CO calc
 
  After relaxation:
    ΔE_coupling = E(slab+C2O2, q=-1) − E(slab+CO, q=0) − E(CO_gas)
    Target: ΔH ≈ −0.19 eV  (Calle-Vallejo & Koper 2013)
 
  Also run q=0 (neutral) for direct comparison with ReaxFF metaD.
  ────────────────────────────────────────────────────────────
""")

pos  [[ 2.55265548  0.         21.715     ]
 [ 1.61418557  0.         22.83342489]
 [ 3.77265548  0.         21.715     ]
 [ 0.85569619  0.         23.73735733]]
pos  [[ 2.55265548  1.817      21.715     ]
 [ 1.61418557  1.817      22.83342489]
 [ 3.77265548  1.817      21.715     ]
 [ 0.85569619  1.817      23.73735733]]
  *C2O2^- dimer — geometry summary (pre-relaxation)
  C–C  bond length         : 1.460 Å   (target 1.46 Å)
  C1–O1 bond length        : 1.220 Å   (target ~1.22 Å)
  C2–O2 bond length        : 1.180 Å   (target ~1.18 Å)
  C1 height above surface  : 1.300 Å   (top site)
  O1 height above surface  : 1.300 Å   (bridge contact, should be < 1 Å)
  C2 height above surface  : 2.418 Å   (upper arm)
  O2 height above surface  : 3.322 Å   (free, toward vacuum)
  Total atoms              : 68
  Fixed atoms (DFT)        : 0

  Bader charge reference (paper Fig. 3b):
    C1 (surface): 2.03 e-  |  O1 (bridge): 7.48 e-
    C2 (upper)  : 3.52 e-  |  O2 (free)  : 7.91 e-
    Sum ≈ 21 e

## 2 CO on the slab

In [ ]:
from ase.build import fcc100
from ase.constraints import FixAtoms
from ase import Atoms
from ase.io import write
import numpy as np

# ── 1. Cu(100) slab 4x4x4 ───────────────────────────────────
slab = fcc100("Cu", size=(4, 4, 4), vacuum=15.0)
slab.center(vacuum=15.0, axis=2)

fixed_ids = [a.index for a in slab if a.position[2] < slab.cell[2][2] / 2 - 5]
slab.set_constraint(FixAtoms(indices=fixed_ids))

a_Cu  = 3.634
z_top = slab.positions[:, 2].max()

# ── 2. Pick two top-site anchors separated by 2x2 lattice ───
# separation vector: (2*a_Cu, 2*a_Cu, 0)
top_atoms  = [a for a in slab if abs(a.position[2] - z_top) < 0.1]
top_sorted = sorted(top_atoms, key=lambda a: (a.position[0], a.position[1]))
anchor1    = top_sorted[5]          # central-ish atom
x1, y1    = anchor1.position[:2]


nn_dist = a_Cu / np.sqrt(2)   # Cu-Cu nearest neighbour on (100)
x2, y2  = x1 + 2*nn_dist, y1 + 2*nn_dist   # (2x2) in primitive surface cell


# ── 3. Two CO molecules, upright on top sites ────────────────
CO_bond  = 1.15   # Å — gas-phase CO (C down, O up)
C_z_above = 1.30  # Å — C height above top Cu layer

CO1 = Atoms("CO", positions=[
    [x1, y1, z_top + C_z_above],
    [x1, y1, z_top + C_z_above + CO_bond],
])

CO2 = Atoms("CO", positions=[
    [x2, y2, z_top + C_z_above],
    [x2, y2, z_top + C_z_above + CO_bond],
])

slab_2co = slab + CO1 + CO2

# ── 4. Sanity checks ─────────────────────────────────────────
n  = len(slab)
p  = slab_2co.positions
C1, O1 = p[n],   p[n+1]
C2, O2 = p[n+2], p[n+3]

cc_dist  = np.linalg.norm(C2 - C1)
co1_dist = np.linalg.norm(O1 - C1)
co2_dist = np.linalg.norm(O2 - C2)

print("=" * 52)
print("  2*CO on Cu(100) — geometry summary")
print("=" * 52)
print(f"  C–C distance (reactant)  : {cc_dist:.3f} Å")
print(f"  CO1 bond length          : {co1_dist:.3f} Å  (target 1.15 Å)")
print(f"  CO2 bond length          : {co2_dist:.3f} Å  (target 1.15 Å)")
print(f"  Anchor separation        : 2x2 lattice = {2*a_Cu:.3f} Å")
print(f"  Total atoms              : {len(slab_2co)}")
print("=" * 52)

assert cc_dist > 3.0, "CO molecules too close — check anchor selection"

# ── 5. Write outputs ─────────────────────────────────────────
write("slab_2CO_reactant.xyz", slab_2co)
write("slab_2CO_reactant.cif", slab_2co)

print("\n  Written: slab_2CO_reactant.xyz")
print("           slab_2CO_reactant.cif")

# view(slab_2co)

## Grepping results

## Adsorption energies, example

### OCCO dimer ads energy

In [2]:
import re
import matplotlib.pyplot as plt

def grep_energy(filename):
    content = open(filename).read()
    return float(re.findall(r"ENERGY\| Total.* (\S+)\n", content)[-1])

!pwd

slab_dimer_name = "slab_C2O2_dimer_top-gopt"
e_slab_dimer = grep_energy(f"{slab_dimer_name}/{slab_dimer_name}.out")
slab_name = "slab-gopt"
e_slab = grep_energy(f"{slab_name}/{slab_name}.out")
co_name = "co-gopt"
e_co = grep_energy(f"{co_name}/{co_name}.out")

ha2eV = 27.21138505

e_ads_dimer = (e_slab_dimer - e_slab - 2*e_co)*ha2eV
print ("E_ads = ",e_ads_dimer," eV")

/home/labuser/BigDrill_bigger
E_ads =  -1.3830280224585647  eV


### CO adsorption energy

In [5]:
import re
import matplotlib.pyplot as plt

def grep_energy(filename):
    content = open(filename).read()
    return float(re.findall(r"ENERGY\| Total.* (\S+)\n", content)[-1])

!pwd

slab_co_name = "slab_CO-gopt"
e_slab_co = grep_energy(f"{slab_co_name}/{slab_co_name}.out")
slab_name = "slab-gopt"
e_slab = grep_energy(f"{slab_name}/{slab_name}.out")
co_name = "co-gopt"
e_co = grep_energy(f"{co_name}/{co_name}.out")

ha2eV = 27.21138505

e_ads_co = (e_slab_co - e_slab - e_co)*ha2eV
print ("E_ads CO = ",e_ads_co," eV")

/home/labuser/BigDrill_bigger
E_ads =  -1.06157773257812  eV


# END

In [16]:

import re
import matplotlib.pyplot as plt

def grep_energy(filename):
    content = open(filename).read()
    return float(re.findall(r"ENERGY\| Total.* (\S+)\n", content)[-1])

!pwd

dim_1_name = "slab_C2O2_dimer_top-charge_1-gopt"
dim_1 = grep_energy(f"{dim_1_name}/{dim_1_name}.out")
dim_0_name = "slab_C2O2_dimer_top-gopt"
dim_0 = grep_energy(f"{dim_0_name}/{dim_0_name}.out")
slab_co_name = "slab_CO-gopt"
slab_co = grep_energy(f"{slab_co_name}/{slab_co_name}.out")
co_name = "co-gopt"
co = grep_energy(f"{co_name}/{co_name}.out")

ha2eV = 27.21138505



/home/labuser/BigDrill_bigger


In [19]:

DG0 = (dim_0 - slab_co - co)*ha2eV
print ("DG(neutral) = ",DG0," eV")

EA = (dim_0 - dim_1)*ha2eV
print ("EA = ",EA," eV")

U_onset = DG0 - EA
print ("U_onset = ",U_onset," eV")

DG(neutral) =  -0.32145028988044494  eV
EA =  1.2557735131622647  eV
U_onset =  -1.5772238030427097  eV


In [6]:
!ls slab_C2O2_dimer_top-charge_1-gopt/slab_C2O2_dimer_top-charge_1-gopt.out


slab_C2O2_dimer_top-charge_1-gopt/slab_C2O2_dimer_top-charge_1-gopt.out
